In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dune_client.client import DuneClient
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries loaded successfully")
print(f"Analysis started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries loaded successfully
Analysis started: 2026-04-22 01:06:21


In [3]:
dune = DuneClient(api_key="Dq0TA4gBF0uBpWrqkG32YuETr9anZVqx")

query_id = 7340955

print("Fetching data from Dune Analytics...")
results = dune.get_latest_result(query_id)
df = pd.DataFrame(results.result.rows)

print(f"Data loaded: {len(df)} sandwich attacks detected")
print(f"Date range: {df['block_time'].min()} to {df['block_time'].max()}")
df.head()

Fetching data from Dune Analytics...
Data loaded: 246 sandwich attacks detected
Date range: 2026-04-17 00:02:23.000 UTC to 2026-04-19 17:54:35.000 UTC


,backrun_tx,block_number,block_time,bot,frontrun_tx,profit_usd,token_pair,victim,victim_amount,victim_tx
0,0x81b04279d073bc6c653e651b4ae17a58c32c18e44979...,24910325,2026-04-19 00:24:59.000 UTC,0x5592bbc09eb109094a9a14b9344c1ca2719c4c64,0x390f39f97a625af064ba005b67b3e82ca023b4b7bb26...,564239.378780,USDT/USDC,0x5592bbc09eb109094a9a14b9344c1ca2719c4c64,32422.030271,0x18f7b0a11d4d334d2944cf7c688f953749a9e5dd2ec8...
1,0x81b04279d073bc6c653e651b4ae17a58c32c18e44979...,24910325,2026-04-19 00:24:59.000 UTC,0x5592bbc09eb109094a9a14b9344c1ca2719c4c64,0x390f39f97a625af064ba005b67b3e82ca023b4b7bb26...,564111.721814,USDC/USDT,0x5592bbc09eb109094a9a14b9344c1ca2719c4c64,32427.864663,0x18f7b0a11d4d334d2944cf7c688f953749a9e5dd2ec8...
2,0x06551cae6973dff103d474aa57d777158b1db14306cd...,24905608,2026-04-18 08:38:47.000 UTC,0x1f2f10d1c40777ae1da742455c65828ff36df387,0x9749c35b1594ff2c8415fcb0b34be7a9dc966aee7413...,285812.376985,USDC/WETH,0x66a9893cc07d91d95644aedd05d03f95e1dba8af,6874.040601,0x849e4c92985fada9e4d1f6af976e373289213c518899...
3,0x06551cae6973dff103d474aa57d777158b1db14306cd...,24905608,2026-04-18 08:38:47.000 UTC,0x1f2f10d1c40777ae1da742455c65828ff36df387,0x9749c35b1594ff2c8415fcb0b34be7a9dc966aee7413...,285812.376985,USDC/WETH,0x66a9893cc07d91d95644aedd05d03f95e1dba8af,46817.487675,0x849e4c92985fada9e4d1f6af976e373289213c518899...
4,0xa1db9bdf1ca0406d96a41b9f7bdb8838cd3ad9a3b25d...,24902399,2026-04-17 21:54:47.000 UTC,0x1f2f10d1c40777ae1da742455c65828ff36df387,0x2c59a0b47853e927d30351243362c0ed2fcb4b58dd93...,157100.511175,USDC/WETH,0x66a9893cc07d91d95644aedd05d03f95e1dba8af,19953.031734,0x562eb79a2e31bcb32ff47df0a8c62c7c0d4aad03ff03...


In [4]:
df['block_time'] = pd.to_datetime(df['block_time'])
df['profit_usd'] = pd.to_numeric(df['profit_usd'])
df['victim_amount'] = pd.to_numeric(df['victim_amount'])

print("="*60)
print("MEV SANDWICH ATTACKS ANALYSIS")
print("="*60)
print(f"\nTotal sandwich attacks detected: {len(df)}")
print(f"Unique MEV bots: {df['bot'].nunique()}")
print(f"Unique victims: {df['victim'].nunique()}")
print(f"\nTotal profit extracted: ${df['profit_usd'].sum():,.2f}")
print(f"Average profit per attack: ${df['profit_usd'].mean():,.2f}")
print(f"Median profit per attack: ${df['profit_usd'].median():,.2f}")
print(f"Max single profit: ${df['profit_usd'].max():,.2f}")
print(f"\nAverage victim trade size: ${df['victim_amount'].mean():,.2f}")
print(f"Total victim trade volume: ${df['victim_amount'].sum():,.2f}")


MEV SANDWICH ATTACKS ANALYSIS

Total sandwich attacks detected: 246
Unique MEV bots: 15
Unique victims: 44

Total profit extracted: $5,008,033.08
Average profit per attack: $20,357.86
Median profit per attack: $902.04
Max single profit: $564,239.38

Average victim trade size: $71,310.54
Total victim trade volume: $17,542,393.84


In [7]:
top_bots = df.groupby('bot').agg({
    'profit_usd': ['sum', 'mean', 'count']
}).round(2)

top_bots.columns = ['total_profit', 'avg_profit', 'attacks']
top_bots = top_bots.sort_values('total_profit', ascending=False).head(10)

print("\nTOP 10 MEV BOTS BY PROFIT")
print("="*60)
print(top_bots)


TOP 10 MEV BOTS BY PROFIT
                                            total_profit  avg_profit  attacks
bot                                                                          
0x1f2f10d1c40777ae1da742455c65828ff36df387    3368507.19    20415.20      165
0x5592bbc09eb109094a9a14b9344c1ca2719c4c64    1372942.54   228823.76        6
0x0906a879ea0f66e3559f11b25b866dba247f9e63     118059.07     5621.86       21
0x03588d14a7aff8a566afd539a4b44cdecca00fdb      57163.39     2858.17       20
0x5814fc20f0e09114503569ed6b27a4a86c9f8de4      28457.21     5691.44        5
0xc1af6f33498692e230b3e6f31a73aeaf0ad0ca3c      27338.42     2102.96       13
0x01fdc48ba0903bb1ae7c517c9287d88ea236f8e1      20682.26     4136.45        5
0xe00456d3af6e9b8715a9c29c88c1932983e064c1       8263.00     8263.00        1
0x654fae4aa229d104cabead47e56703f58b174be4       2926.44     2926.44        1
0xa462d9acaccb141ce7f17213b95198fe248c27a1       1330.75      665.37        2


In [6]:
df['hour'] = df['block_time'].dt.hour
df['date'] = df['block_time'].dt.date

hourly_stats = df.groupby('hour').agg({
    'profit_usd': ['sum', 'mean', 'count']
}).round(2)

hourly_stats.columns = ['total_profit', 'avg_profit', 'attacks']
hourly_stats = hourly_stats.reset_index()

print("\nMEV ACTIVITY BY HOUR (UTC)")
print("="*60)
print(hourly_stats)

print(f"\nMost active hour: {hourly_stats.loc[hourly_stats['attacks'].idxmax(), 'hour']}:00 UTC ({hourly_stats['attacks'].max()} attacks)")
print(f"Most profitable hour: {hourly_stats.loc[hourly_stats['total_profit'].idxmax(), 'hour']}:00 UTC (${hourly_stats['total_profit'].max():,.2f})")


MEV ACTIVITY BY HOUR (UTC)
    hour  total_profit  avg_profit  attacks
0      0    1373149.41   171643.68        8
1      1        114.72      114.72        1
2      2       1405.15      468.38        3
3      3        248.98      124.49        2
4      4      79583.46     5305.56       15
5      5      23686.09     2153.28       11
6      6       5327.11      532.71       10
7      7      50495.41    10099.08        5
8      8    2693458.75    51797.28       52
9      9      55720.59     7960.08        7
10    10      19098.80     1061.04       18
11    11       1364.40      272.88        5
12    12       5210.62     1736.87        3
13    13       4561.21      304.08       15
14    14      44591.23     2346.91       19
15    15       5249.72      874.95        6
16    16       4191.77      523.97        8
17    17     110848.12     8526.78       13
18    18       2341.21     1170.60        2
19    19      25792.23     2865.80        9
20    20      13413.70     1676.71        8
21  

In [8]:
df['token_pair_clean'] = df['token_pair'].str.replace('/', '_')

token_stats = df.groupby('token_pair').agg({
    'profit_usd': ['sum', 'mean', 'count'],
    'victim_amount': 'mean'
}).round(2)

token_stats.columns = ['total_profit', 'avg_profit', 'attacks', 'avg_victim_amount']
token_stats = token_stats.sort_values('attacks', ascending=False).head(10)

print("\nTOP 10 TOKEN PAIRS BY ATTACK FREQUENCY")
print("="*60)
print(token_stats)


TOP 10 TOKEN PAIRS BY ATTACK FREQUENCY
            total_profit  avg_profit  attacks  avg_victim_amount
token_pair                                                      
USDT/USDC     2748627.31    29555.13       93          159667.60
USDT/WETH      265639.83     5651.91       47           12708.68
USDC/WETH     1052911.19    23929.80       44           16143.28
WETH/USDT      152540.26     4237.23       36           10043.41
USDC/USDT      696546.38    58045.53       12           65690.31
WETH/USDC       91353.71     8304.88       11           12257.72
DAI/WETH          178.42      178.42        1            8098.42
USDC/DAI          103.34      103.34        1           45646.35
USDT/DAI          132.63      132.63        1           47269.10


In [9]:
df_export = df.copy()

df_export['bot_short'] = df_export['bot'].str[:8] + '...' + df_export['bot'].str[-6:]
df_export['victim_short'] = df_export['victim'].str[:8] + '...' + df_export['victim'].str[-6:]

export_columns = [
    'block_number', 'block_time', 'date', 'hour',
    'bot', 'bot_short', 'victim', 'victim_short',
    'token_pair', 'victim_amount', 'profit_usd',
    'frontrun_tx', 'victim_tx', 'backrun_tx'
]

df_export = df_export[export_columns]

df_export.to_csv('../data/mev_sandwich_data.csv', index=False)

print(f"\nData exported: {len(df_export)} rows")
print(f"File saved: data/mev_sandwich_data.csv")
print(f"\nColumns exported: {list(df_export.columns)}")



Data exported: 246 rows
File saved: data/mev_sandwich_data.csv

Columns exported: ['block_number', 'block_time', 'date', 'hour', 'bot', 'bot_short', 'victim', 'victim_short', 'token_pair', 'victim_amount', 'profit_usd', 'frontrun_tx', 'victim_tx', 'backrun_tx']


In [10]:
bot_stats = df.groupby('bot').agg({
    'profit_usd': ['sum', 'mean', 'median', 'max'],
    'victim_amount': 'mean',
    'block_number': 'count'
}).round(2)

bot_stats.columns = ['total_profit', 'avg_profit', 'median_profit', 'max_profit', 'avg_victim_amount', 'attack_count']
bot_stats = bot_stats.reset_index()
bot_stats['bot_short'] = bot_stats['bot'].str[:8] + '...' + bot_stats['bot'].str[-6:]
bot_stats.to_csv('../data/mev_bots_stats.csv', index=False)

hourly_export = df.groupby(['date', 'hour']).agg({
    'profit_usd': ['sum', 'mean', 'count'],
    'victim_amount': 'sum'
}).round(2)

hourly_export.columns = ['total_profit', 'avg_profit', 'attack_count', 'total_victim_volume']
hourly_export = hourly_export.reset_index()
hourly_export.to_csv('../data/mev_hourly_stats.csv', index=False)

token_export = df.groupby('token_pair').agg({
    'profit_usd': ['sum', 'mean', 'count'],
    'victim_amount': ['sum', 'mean']
}).round(2)

token_export.columns = ['total_profit', 'avg_profit', 'attack_count', 'total_victim_volume', 'avg_victim_amount']
token_export = token_export.reset_index()
token_export.to_csv('../data/mev_token_stats.csv', index=False)

print("Additional datasets exported:")
print(f"- mev_bots_stats.csv ({len(bot_stats)} bots)")
print(f"- mev_hourly_stats.csv ({len(hourly_export)} time periods)")
print(f"- mev_token_stats.csv ({len(token_export)} token pairs)")

Additional datasets exported:
- mev_bots_stats.csv (15 bots)
- mev_hourly_stats.csv (51 time periods)
- mev_token_stats.csv (9 token pairs)


In [11]:
print("\n" + "="*70)
print("MEV SANDWICH ATTACKS - FINAL SUMMARY REPORT")
print("="*70)

print(f"\nDATA OVERVIEW")
print("-"*70)
print(f"Analysis period: {df['block_time'].min()} to {df['block_time'].max()}")
print(f"Total sandwich attacks: {len(df)}")
print(f"Unique MEV bots: {df['bot'].nunique()}")
print(f"Unique victims: {df['victim'].nunique()}")

print(f"\nFINANCIAL IMPACT")
print("-"*70)
print(f"Total MEV profit extracted: ${df['profit_usd'].sum():,.2f}")
print(f"Total victim trade volume: ${df['victim_amount'].sum():,.2f}")
print(f"MEV extraction rate: {(df['profit_usd'].sum() / df['victim_amount'].sum() * 100):.2f}%")
print(f"Average profit per attack: ${df['profit_usd'].mean():,.2f}")
print(f"Median profit per attack: ${df['profit_usd'].median():,.2f}")
print(f"Largest single attack profit: ${df['profit_usd'].max():,.2f}")

print(f"\nTOP PERFORMER")
print("-"*70)
top_bot = top_bots.index[0]
print(f"Most profitable bot: {top_bot[:10]}...{top_bot[-8:]}")
print(f"Total profit: ${top_bots.iloc[0]['total_profit']:,.2f}")
print(f"Number of attacks: {int(top_bots.iloc[0]['attacks'])}")
print(f"Average profit per attack: ${top_bots.iloc[0]['avg_profit']:,.2f}")

print(f"\nTIME ANALYSIS")
print("-"*70)
peak_hour = hourly_stats.loc[hourly_stats['attacks'].idxmax()]
print(f"Peak activity hour: {int(peak_hour['hour'])}:00 UTC")
print(f"Attacks in peak hour: {int(peak_hour['attacks'])}")
print(f"Profit in peak hour: ${peak_hour['total_profit']:,.2f}")

print(f"\nTOKEN ANALYSIS")
print("-"*70)
top_pair = token_stats.index[0]
print(f"Most targeted pair: {top_pair}")
print(f"Attacks on this pair: {int(token_stats.iloc[0]['attacks'])}")
print(f"Total profit from this pair: ${token_stats.iloc[0]['total_profit']:,.2f}")

print(f"\nEXPORTED FILES")
print("-"*70)
print("1. data/mev_sandwich_data.csv - Raw sandwich attack data")
print("2. data/mev_bots_stats.csv - Bot performance statistics")
print("3. data/mev_hourly_stats.csv - Hourly time series data")
print("4. data/mev_token_stats.csv - Token pair analysis")

print("\n" + "="*70)
print("Analysis completed successfully!")
print("="*70 + "\n")


MEV SANDWICH ATTACKS - FINAL SUMMARY REPORT

DATA OVERVIEW
----------------------------------------------------------------------
Analysis period: 2026-04-17 00:02:23+00:00 to 2026-04-19 17:54:35+00:00
Total sandwich attacks: 246
Unique MEV bots: 15
Unique victims: 44

FINANCIAL IMPACT
----------------------------------------------------------------------
Total MEV profit extracted: $5,008,033.08
Total victim trade volume: $17,542,393.84
MEV extraction rate: 28.55%
Average profit per attack: $20,357.86
Median profit per attack: $902.04
Largest single attack profit: $564,239.38

TOP PERFORMER
----------------------------------------------------------------------
Most profitable bot: 0x1f2f10d1...f36df387
Total profit: $3,368,507.19
Number of attacks: 165
Average profit per attack: $20,415.20

TIME ANALYSIS
----------------------------------------------------------------------
Peak activity hour: 8:00 UTC
Attacks in peak hour: 52
Profit in peak hour: $2,693,458.75

TOKEN ANALYSIS
------

In [13]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

st.set_page_config(page_title="MEV Sandwich Attack Monitor", layout="wide")

st.title("🥪 MEV Sandwich Attack Monitor")
st.markdown("Real-time analysis of Maximal Extractable Value (MEV) sandwich attacks on Ethereum")

@st.cache_data
def load_data():
    df = pd.read_csv('../data/mev_sandwich_data.csv')
    df['block_time'] = pd.to_datetime(df['block_time'])
    return df

df = load_data()

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("Total Attacks", f"{len(df):,}")
    
with col2:
    st.metric("Total Profit", f"${df['profit_usd'].sum():,.0f}")
    
with col3:
    st.metric("Unique Bots", df['bot'].nunique())
    
with col4:
    st.metric("Avg Profit", f"${df['profit_usd'].mean():,.0f}")

st.markdown("---")

st.subheader("Filter Data")

col1, col2 = st.columns(2)

with col1:
    selected_tokens = st.multiselect(
        "Token Pairs",
        options=df['token_pair'].unique(),
        default=df['token_pair'].unique()
    )
    
with col2:
    min_profit = st.slider(
        "Minimum Profit (USD)",
        min_value=0,
        max_value=int(df['profit_usd'].max()),
        value=0,
        step=1000
    )

filtered_df = df[
    (df['token_pair'].isin(selected_tokens)) &
    (df['profit_usd'] >= min_profit)
]

st.markdown(f"**Showing {len(filtered_df)} attacks**")

st.markdown("---")

col1, col2 = st.columns(2)

with col1:
    st.subheader("Profit Over Time")
    
    hourly_profit = filtered_df.groupby('hour')['profit_usd'].sum().reset_index()
    
    fig1 = px.bar(
        hourly_profit,
        x='hour',
        y='profit_usd',
        labels={'hour': 'Hour (UTC)', 'profit_usd': 'Total Profit (USD)'},
        color='profit_usd',
        color_continuous_scale='Reds'
    )
    fig1.update_layout(showlegend=False, height=400)
    st.plotly_chart(fig1, use_container_width=True)

with col2:
    st.subheader("Attacks by Token Pair")
    
    token_counts = filtered_df['token_pair'].value_counts().reset_index()
    token_counts.columns = ['token_pair', 'count']
    
    fig2 = px.pie(
        token_counts,
        values='count',
        names='token_pair',
        hole=0.4
    )
    fig2.update_layout(height=400)
    st.plotly_chart(fig2, use_container_width=True)

st.markdown("---")

st.subheader("Top MEV Bots")

bot_stats = filtered_df.groupby('bot_short').agg({
    'profit_usd': ['sum', 'mean', 'count']
}).round(2)

bot_stats.columns = ['Total Profit', 'Avg Profit', 'Attacks']
bot_stats = bot_stats.sort_values('Total Profit', ascending=False).head(10)

st.dataframe(bot_stats, use_container_width=True)

st.markdown("---")

st.subheader("Recent Attacks")

recent = filtered_df.sort_values('block_time', ascending=False).head(20)

display_cols = ['block_time', 'bot_short', 'victim_short', 'token_pair', 'profit_usd', 'victim_amount']
st.dataframe(recent[display_cols], use_container_width=True)

st.markdown("---")
st.caption("Data source: Dune Analytics | Built with Streamlit")

2026-04-22 01:30:09.120 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 01:30:09.121 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 01:30:09.122 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 01:30:09.122 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 01:30:09.123 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 01:30:09.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 01:30:09.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 01:30:09.125 No runtime found, using MemoryCacheStorageManager
2026-04-22 01:30:09.126 Thread 'MainThread':

DeltaGenerator()